In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Once mounted, your file paths in the config will change to something like:
# METADATA_PATH = "/content/drive/MyDrive/FIT3164/metadata_holistic_pose_hand.csv"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os
import json
import zipfile
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

CONFIG (COLAB VERSION)

In [ ]:
ZIP_PATH = "/content/drive/MyDrive/FIT3164/train_dataset_v1/v1.zip"
METADATA_PATH = "/content/drive/MyDrive/FIT3164/train_dataset_v1/mdv1.csv"

EXTRACT_DIR = "/content"
NPY_DIR = "/content/v1"
LABEL_MAP_PATH = "/content/label_map_v1.json"
MODEL_SAVE_PATH = "/content/best_sign_lstm_v1.pth"

BATCH_SIZE = 32
NUM_EPOCHS = 20
LEARNING_RATE = 0.001
MIN_VALID_FRAMES = 5
INPUT_SIZE = 214
HIDDEN_SIZE = 128
NUM_LAYERS = 2

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)

Using device: cpu


UNZIP DATA

In [ ]:
if not os.path.exists(NPY_DIR):
    with zipfile.ZipFile(ZIP_PATH, "r") as zip_ref:
        zip_ref.extractall(EXTRACT_DIR)
    print("Zip extracted.")
else:
    print("Npy folder already exists, skipping unzip.")


Npy folder already exists, skipping unzip.


DATASET

In [ ]:
class WLASLDataset(Dataset):
    def __init__(self, df):
        self.df = df.reset_index(drop=True)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        x = np.load(row["colab_path"]).astype(np.float32)   # (30, 214)

        if x.shape != (30, 214):
            raise ValueError(f"Expected shape (30, 214), got {x.shape} for file {row['colab_path']}")

        y = int(row["label_id"])

        x = torch.tensor(x, dtype=torch.float32)
        y = torch.tensor(y, dtype=torch.long)

        return x, y

MODEL

In [ ]:
class SignLSTM(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, num_classes):
        super().__init__()

        self.cnn = nn.Sequential(
            nn.Conv1d(
                in_channels=input_size,
                out_channels=input_size,
                kernel_size=3,
                padding=1,
            ),
            nn.BatchNorm1d(input_size),
            nn.ReLU(),
        )

        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=0.3 if num_layers > 1 else 0.0,
            bidirectional=True,
        )

        self.layer_norm = nn.LayerNorm(hidden_size * 2)

        self.attention = nn.Sequential(
            nn.Linear(hidden_size * 2, hidden_size),
            nn.Tanh(),
            nn.Linear(hidden_size, 1)
        )

        self.fc = nn.Linear(hidden_size * 2, num_classes)

    def forward(self, x):
        # x: (B, 30, 214)
        x = x.transpose(1, 2)   # (B, 214, 30)
        x = self.cnn(x)
        x = x.transpose(1, 2)   # (B, 30, 214)

        lstm_out, _ = self.lstm(x)   # (B, 30, hidden_size * 2)
        lstm_out = self.layer_norm(lstm_out)

        attn_scores = self.attention(lstm_out)      # (B, 30, 1)
        attn_weights = torch.softmax(attn_scores, dim=1)
        context = torch.sum(attn_weights * lstm_out, dim=1)  # (B, hidden_size * 2)

        logits = self.fc(context)
        return logits

TRAIN / EVAL FUNCTIONS

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()

    total_loss = 0.0
    total_correct = 0
    total_samples = 0

    for x, y in loader:
        x = x.to(device)
        y = y.to(device)

        optimizer.zero_grad()
        logits = model(x)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * x.size(0)
        preds = torch.argmax(logits, dim=1)
        total_correct += (preds == y).sum().item()
        total_samples += x.size(0)

    avg_loss = total_loss / total_samples
    avg_acc = total_correct / total_samples
    return avg_loss, avg_acc


def evaluate(model, loader, criterion, device):
    model.eval()

    total_loss = 0.0
    total_correct = 0
    total_samples = 0

    with torch.no_grad():
        for x, y in loader:
            x = x.to(device)
            y = y.to(device)

            logits = model(x)
            loss = criterion(logits, y)

            total_loss += loss.item() * x.size(0)
            preds = torch.argmax(logits, dim=1)
            total_correct += (preds == y).sum().item()
            total_samples += x.size(0)

    avg_loss = total_loss / total_samples
    avg_acc = total_correct / total_samples
    return avg_loss, avg_acc

MAIN

In [ ]:
def main():
    # 1. Read metadata
    df = pd.read_csv(METADATA_PATH, dtype={"video_id": str})

    print("Original samples:", len(df))
    print(df.head())
    print()

    # 2. Filter low-quality samples
    df = df[df["valid_frames"] >= MIN_VALID_FRAMES].reset_index(drop=True)

    print("After filtering valid_frames >=", MIN_VALID_FRAMES, ":", len(df))
    print("Unique gloss:", df["gloss"].nunique())
    print()
    print("Split counts:")
    print(df["split"].value_counts())
    print()

    # 3. Rebuild Colab path
    df["colab_path"] = df["video_id"].astype(str).apply(
        lambda x: os.path.join(NPY_DIR, f"{x}.npy")
    )

    # 4. Keep only files that actually exist
    df = df[df["colab_path"].apply(os.path.exists)].reset_index(drop=True)

    print("After checking existing npy files:", len(df))
    print()

    # 5. Create label mapping
    glosses = sorted(df["gloss"].unique())
    label2id = {gloss: i for i, gloss in enumerate(glosses)}
    id2label = {i: gloss for gloss, i in label2id.items()}

    df["label_id"] = df["gloss"].map(label2id)

    with open(LABEL_MAP_PATH, "w", encoding="utf-8") as f:
        json.dump(
            {
                "label2id": label2id,
                "id2label": id2label
            },
            f,
            ensure_ascii=False,
            indent=2
        )

    num_classes = len(label2id)
    print("Number of classes:", num_classes)
    print()

    # 6. Split dataset
    train_df = df[df["split"] == "train"].reset_index(drop=True)
    val_df = df[df["split"] == "val"].reset_index(drop=True)
    test_df = df[df["split"] == "test"].reset_index(drop=True)

    print("Train:", len(train_df))
    print("Val:", len(val_df))
    print("Test:", len(test_df))
    print()

    # 7. Create dataset / dataloader
    train_dataset = WLASLDataset(train_df)
    val_dataset = WLASLDataset(val_df)
    test_dataset = WLASLDataset(test_df)

    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
    test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

    # Check one batch
    x_sample, y_sample = next(iter(train_loader))
    print("One batch x shape:", x_sample.shape)
    print("One batch y shape:", y_sample.shape)
    print()

    # 8. Build model
    model = SignLSTM(
        input_size=INPUT_SIZE,
        hidden_size=HIDDEN_SIZE,
        num_layers=NUM_LAYERS,
        num_classes=num_classes
    ).to(DEVICE)

    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

    print("Using device:", DEVICE)
    print()

    # 9. Training
    best_val_acc = 0.0

    for epoch in range(NUM_EPOCHS):
        train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, DEVICE)
        val_loss, val_acc = evaluate(model, val_loader, criterion, DEVICE)

        print(f"Epoch {epoch + 1}/{NUM_EPOCHS}")
        print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
        print(f"Val   Loss: {val_loss:.4f} | Val   Acc: {val_acc:.4f}")
        print("-" * 50)

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save(model.state_dict(), MODEL_SAVE_PATH)
            print("Best model saved.")
            print("-" * 50)

    # 10. Final test
    print("Loading best model for final test...")
    model.load_state_dict(torch.load(MODEL_SAVE_PATH, map_location=DEVICE))

    test_loss, test_acc = evaluate(model, test_loader, criterion, DEVICE)
    print(f"Test Loss: {test_loss:.4f} | Test Acc: {test_acc:.4f}")

In [ ]:
if __name__ == "__main__":
    main()

Original samples: 59
  video_id  gloss  split                                               path  \
0    69302  drink    val  D:\Monash University\Monash 2026 Sem 1\FIT 316...   
1    65539  drink  train  D:\Monash University\Monash 2026 Sem 1\FIT 316...   
2    17710  drink  train  D:\Monash University\Monash 2026 Sem 1\FIT 316...   
3    17733  drink  train  D:\Monash University\Monash 2026 Sem 1\FIT 316...   
4    65540  drink  train  D:\Monash University\Monash 2026 Sem 1\FIT 316...   

   valid_frames  
0            30  
1            30  
2            30  
3            29  
4            30  

After filtering valid_frames >= 5 : 59
Unique gloss: 5

Split counts:
split
train    42
val      10
test      7
Name: count, dtype: int64

After checking existing npy files: 59

Number of classes: 5

Train: 42
Val: 10
Test: 7

One batch x shape: torch.Size([32, 30, 214])
One batch y shape: torch.Size([32])

Using device: cpu

Epoch 1/20
Train Loss: 1.5992 | Train Acc: 0.2857
Val   Loss: 1.837

In [ ]:
import zipfile
import os

with zipfile.ZipFile(ZIP_PATH, 'r') as zip_ref:
    zip_ref.extractall(EXTRACT_DIR)

print("Unzip done.")
print(os.listdir("/content")[:20])
print(os.listdir("/content/v1")[:20])

Unzip done.
['.config', 'label_map_v1.json', 'v1', '.ipynb_checkpoints', 'best_sign_lstm_v1.pth', 'drive', 'sample_data']
['66606.npy', '12316.npy', '65167.npy', '12338.npy', '63230.npy', '63227.npy', '17712.npy', '63229.npy', '57939.npy', '05727.npy', '05733.npy', '57943.npy', '66607.npy', '57935.npy', '17733.npy', '69534.npy', '57940.npy', '12320.npy', '63231.npy', '63228.npy']


In [ ]:
df = pd.read_csv(METADATA_PATH, dtype={"video_id": str})

df["colab_path"] = df["video_id"].astype(str).apply(
    lambda x: os.path.join(NPY_DIR, f"{x}.npy")
)

print("NPY_DIR:", NPY_DIR)
print("NPY_DIR exists:", os.path.exists(NPY_DIR))
print("Files in NPY_DIR:", len(os.listdir(NPY_DIR)))
print(df[["video_id", "colab_path"]].head(10))
print("Existing files:", df["colab_path"].apply(os.path.exists).sum())
print("Missing files:", (~df["colab_path"].apply(os.path.exists)).sum())

NPY_DIR: /content/v1
NPY_DIR exists: True
Files in NPY_DIR: 59
  video_id             colab_path
0    69302  /content/v1/69302.npy
1    65539  /content/v1/65539.npy
2    17710  /content/v1/17710.npy
3    17733  /content/v1/17733.npy
4    65540  /content/v1/65540.npy
5    17734  /content/v1/17734.npy
6    17711  /content/v1/17711.npy
7    17712  /content/v1/17712.npy
8    17713  /content/v1/17713.npy
9    17709  /content/v1/17709.npy
Existing files: 59
Missing files: 0
